###Ingest npci parquet file   
1.read file using dataframe reader api   
2.add metadata columns - source file,ingestion timestamp     
3.write bronze tables using dataframe write api

In [0]:
dbutils.widgets.text("p_batch_date","")
v_batch_date = dbutils.widgets.get("p_batch_date")

In [0]:
%run ../00-common/01.environment_config


In [0]:
%run ../00-common/02.bronze_functions

In [0]:
source_file = f"{raw_path}/{v_batch_date}/npci/"

In [0]:
table_name = f"{catalog_name}.{bronze_schema}.npci"

In [0]:
from pyspark.sql.types import *

In [0]:
npci_schema = StructType([
  StructField('TxnID', StringType()),
  StructField('BillerID', StringType()),
  StructField('CustomerID', StringType()),
  StructField('TxnAmount', DoubleType()),
  StructField('Status', StringType()),
  StructField('npcimentCycleID', StringType()),
  StructField('Timestamp', TimestampNTZType())
])

In [0]:
npci_df = spark.read.format('parquet') \
                .option('mode', 'FAILFAST') \
                .load(source_file)

In [0]:
npci_audit = add_ingestion_metadata(npci_df)

In [0]:
npci_final = npci_audit.withColumn("batch_date", F.lit(v_batch_date))

In [0]:
npci_final.write.mode('overwrite').partitionBy('batch_date').option('replaceWhere',f"batch_date = '{v_batch_date}'").saveAsTable(table_name)

In [0]:
display(spark.table(table_name))

TxnID,BillerID,ConsumerNumber,TxnAmount,SettlementStatus,BillerRefID,SettlementDate,ingestion_timestamp,source_file,batch_date
TXN1001,KSEB,CONS001,1500.0,Settled,BREF100001,2026-08-07,2026-08-22T13:21:07.893Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/npci/app_20260807_001.parquet,2026-08-13
TXN1002,WTR01,CONS002,800.0,Settled,BREF100002,2026-08-07,2026-08-22T13:21:07.893Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/npci/app_20260807_001.parquet,2026-08-13
TXN1003,TEL01,CONS003,999.0,Pending,null,2026-08-07,2026-08-22T13:21:07.893Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/npci/app_20260807_001.parquet,2026-08-13
TXN1004,GAS01,CONS004,1200.0,Failed,null,2026-08-07,2026-08-22T13:21:07.893Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/npci/app_20260807_001.parquet,2026-08-13
TXN1005,DTH01,CONS005,450.0,Settled,BREF100005,2026-08-07,2026-08-22T13:21:07.893Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/npci/app_20260807_001.parquet,2026-08-13
TXN1006,KSEB,CONS006,2300.0,Settled,BREF100006,2026-08-07,2026-08-22T13:21:07.893Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/npci/app_20260807_002.parquet,2026-08-13
TXN1007,GAS01,CONS007,1750.0,Settled,BREF100007,2026-08-07,2026-08-22T13:21:07.893Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/npci/app_20260807_002.parquet,2026-08-13
TXN1008,TEL01,CONS008,699.0,Settled,BREF100008,2026-08-07,2026-08-22T13:21:07.893Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/npci/app_20260807_002.parquet,2026-08-13
TXN1009,WTR01,CONS009,650.0,Pending,null,2026-08-07,2026-08-22T13:21:07.893Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/npci/app_20260807_002.parquet,2026-08-13
TXN1010,DTH01,CONS010,399.0,Settled,BREF100010,2026-08-07,2026-08-22T13:21:07.893Z,dbfs:/Volumes/payment_app/raw/files/2026-08-13/npci/app_20260807_002.parquet,2026-08-13
